# Run the local VLM arm on a free Colab T4 (IT-98)

The dual-extractor pipeline needs two independent readings of each card.
Gemini runs anywhere; `qwen2.5-vl` needs a real GPU, which nobody's laptop
has. This notebook is that GPU.

It produces one thing: a `raw/` directory of the model's answers, written
into the shared Drive. `spike/dual_extract.py` pairs it against Gemini's on
any machine afterwards — the pairing reads directories, not live models, so
the two arms never have to run in the same place.

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

Free tier gives ~12h per session and disconnects after ~90 min idle. Thirty
cards is well inside both, but leave the tab open.

In [ ]:
# Fail now, not eight minutes into a model download, if there is no GPU.
import torch

assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU"
name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"{name}, {vram:.0f} GB, bf16 supported: {torch.cuda.is_bf16_supported()}")
print("-> compute dtype will be", "bfloat16" if torch.cuda.is_bf16_supported() else "float16")

In [ ]:
# Colab ships torch; the rest is ours. accelerate is what device_map="auto"
# needs, bitsandbytes is the 4-bit loader.
%pip install -q accelerate bitsandbytes qwen-vl-utils openpyxl pypdf "transformers>=4.57"

## The repo

Private, so the clone needs a token. `getpass` keeps it off the screen and
out of the saved notebook — **do not** paste a token into a cell instead.
Use a fine-grained PAT with read-only contents access, and delete it when
you are done.

The token is used once, for the clone, and not written anywhere.

In [ ]:
import getpass
import subprocess
from pathlib import Path

REPO = "skandaramanan/IT-Project-T8-6"
BRANCH = "skandar/it-98-wire-the-hosted-second-extractor-into-the-harness-with"
checkout = Path("/content/IT-Project-T8-6")

if not checkout.exists():
    token = getpass.getpass("GitHub token (input hidden): ")
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH,
         f"https://{token}@github.com/{REPO}.git", str(checkout)],
        check=True,
    )
    del token
print("checked out:", checkout)

## The data — upload this once, first

The Drive holds the **raw scans** (`Wilfred_Prest_Survey_Data/`, the numbered
unit folders). It does **not** hold the C1/C2 benchmark set — the 30 selected
cards and their hand-checked sheet live only on whoever ran
`select_c1_c2_sample.py`.

So before running this notebook, copy your local `data/benchmark/c1c2/`
(146 MB: 30 PNGs, `coding-card-c1-c2-ground-truth.xlsx`, `manifest.csv`)
into Drive, keeping the folder name. Both arms then score against the same
30 cards, which is the only way their agreement means anything.

Set `DRIVE_ROOT` to wherever you put it.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# Wherever the shared folder lands for you -- a Shared Drive is usually
# /content/drive/Shareddrives/<name>, your own copy /content/drive/MyDrive/<name>.
DRIVE_ROOT = Path("/content/drive/MyDrive/IT-Project")

BENCHMARK = DRIVE_ROOT / "c1c2"
GROUND_TRUTH = BENCHMARK / "coding-card-c1-c2-ground-truth.xlsx"
IMAGES = BENCHMARK

# Results go back to Drive, not /content -- Colab's disk dies with the
# session, and this raw/ directory is the entire point of the run.
RESULTS_ROOT = DRIVE_ROOT / "results"

if not GROUND_TRUTH.exists():
    raise SystemExit(
        f"No benchmark sheet at {GROUND_TRUTH}.\n"
        "Upload your local data/benchmark/c1c2/ to Drive and point DRIVE_ROOT at it."
    )

images = sorted(IMAGES.glob("*.png"))
print(f"sheet : {GROUND_TRUTH.name}")
print(f"images: {len(images)} png (expect 30)")
print(f"results -> {RESULTS_ROOT}")

## One card first

The model is a ~16GB download and the first card pays for it. Run one before
committing to thirty: it proves the weights load 4-bit, the image fits, and
the JSON parses. If this OOMs, drop `MAX_PIXELS` in
`spike/qwen_vl_extractor.py` rather than reaching for a bigger GPU.

In [ ]:
%cd /content/IT-Project-T8-6
import sys

sys.path[:0] = ["src", "spike"]
from ground_truth import load_ground_truth

first = load_ground_truth(GROUND_TRUTH, IMAGES)[0]
print("first card:", first.page_id, "->", first.image_path.name)

!python spike/run_benchmark.py --model qwen2.5-vl \
  --ground-truth "{GROUND_TRUTH}" --images "{IMAGES}" \
  --results-root "{RESULTS_ROOT}" --workers 1 --pages {first.page_id}

## The rest

`--workers 1` because the GPU is the bottleneck, not the network — parallel
workers on one card would just fight over VRAM.

Cards already answered are skipped, so a disconnect costs only what was in
flight. Re-run this cell to pick up where it stopped.

In [ ]:
!python spike/run_benchmark.py --model qwen2.5-vl \
  --ground-truth "{GROUND_TRUTH}" --images "{IMAGES}" \
  --results-root "{RESULTS_ROOT}" --workers 1

In [ ]:
# What made it into Drive. This is the artefact -- the accuracy number above
# is IT-100's business, not this notebook's.
raw = RESULTS_ROOT / "qwen2.5-vl" / "raw"
answered = [p for p in sorted(raw.glob("*.txt")) if p.stat().st_size]
expected = {c.page_id for c in load_ground_truth(GROUND_TRUTH, IMAGES)}

print(f"{len(answered)}/{len(expected)} cards answered, saved to {raw}")
missing = sorted(expected - {p.stem for p in answered})
print("still unread:", missing or "none")

## Back on your own machine

Sync the Drive `results/` folder down, then pair the two arms:

```bash
uv run python spike/dual_extract.py --models gemini qwen2.5-vl --no-fetch
```

`--no-fetch` because both arms are already extracted — one here, one there —
and there is nothing left to buy.